# Forward and reverse-mode dynamics of a birefringent multimode fiber

The goal of this notebook and the other notebooks in this folder is to show an example of the usage of this code and various features. This notebook demonstrates the public PulsePropagation API on a retained GRIN-fiber example using CSV fiber assets bundled with the release. 

The first example demonstrates forward solving and gradient solving for a birefringent multimode soliton.

For a compact public run, this notebook makes two adjustments: I make the fiber shorter (to reduce overall simulation time and to make autodifferentiation feasible) and I do not multiply each mode (spatial + polarization) by random phases as to make a more easily reproducible output.

Loading packages below. 

In [ ]:
using Pkg

const ROOT = abspath(joinpath(@__DIR__, ".."))
Pkg.activate(ROOT)

using FFTW
using LinearAlgebra
using PyPlot
using Printf

isdefined(Main, :PulsePropagation) ||
    include(joinpath(ROOT, "src", "PulsePropagation.jl"))
using .PulsePropagation

const c_light_m_ps = 2.99792458e-4;

Let's instantiate basic simulation parameters.

In [ ]:
lambda0_m = 1550e-9
f0_THz = c_light_m_ps / lambda0_m;

Nt = 2^11
time_window_ps = 50.0
grid = TimeGrid(Nt, time_window_ps)

t = time_axis(grid);
f = f0_THz .+ frequency_axis(grid; shifted=true);
λ = wavelength_axis(grid, f0_THz; shifted=true);

Let's load and build the fiber.

In [ ]:
## Grid, fiber, solver, and two-mode Gaussian launch matching notebooks/gif625_opt.ipynb

fiber_folder = joinpath(ROOT, "GRIN_62.5um_wavelength1550nm_csv");

num_modes = 2
L_fiber = 2;

# To implement birefringence, we will do something hacky since the fiber is nomimally scalar. We will
# load a beta matrix for a scalar system, expand it by a factor of 2, and manipulate one polarization
# relative to another.

system, dofs = load_fiber_system(
    fiber_folder;
    modes=1:num_modes,
    length=L_fiber,
    lambda0=lambda0_m,
    material=Silica(),
    betas_filename="betas.csv",
    s_tensors_filename="S_tensors_10modes.csv",
        scalar=false,
    raman_fraction=0.245,
)

## Using values of beta from wise example. 

betas_biref = [5.847462729085851e+06	5.851435323667164e+06	5.841335866995176e+06	5.845308461576490e+06
4.878764128935511e+03	0	4.878836422580576e+03	0
-0.028100040576544	0	-0.028248131742058	0
1.518979580892188e-04	0	1.524304694210461e-04	0
-4.917494281128627e-07	0	-4.940312405234015e-07	0
2.315909039461211e-09	0	2.327945659939789e-09	0];

system_biref = PassiveFiber(;
    length = system.length,
    lambda0 = system.lambda0,
    f0 = system.f0,
    material = system.material,
    betas = betas_biref ,
    sr = system.geometry.sr,
    n2 = system.n2,
    raman_fraction = system.raman_fraction,
    source_path = system.source_path,
)

dofs = PolarizedModalField(1:2);


Setup an initial field. Here, the same pulse in each mode.

In [ ]:
tfwhm_ps = 0.500
peak_power_W = 9395; # close enough to 5 nJ.
mode_coefficients = ComplexF64[1,1,1,1]

fields = gaussian_pulse(
    grid;
    fwhm=tfwhm_ps,
    peak_power=peak_power_W,
    dofs=dofs,
    coefficients=mode_coefficients,
)

println("The pulse energy is $(sum(abs2.(fields[:,:,:]))*(t[2]-t[1])) pJ")

Below we setup the relevant terms we want to consider in our model, instantiate a solver, setup a "problem" and solve the forward PulsePropagation. The lines after construct and plot the spectrum.

In [ ]:
terms = PropagationTerms(Dispersion(), Kerr(), Raman(:model))

# solver = RK4IPSolver(;
#     stepping=FixedStep(dz = 1e-2),
#     saveat=SaveAt([0.0, system_biref.length]),
#     reltol=1e-6,
#     abstol=1e-6,
# )

solver = RK4IPSolver(;
    stepping = AdaptiveStep(
        initial_dz = 1e-2,
        max_dz = 1e-1,
        rtol = 1e-8,
        atol = 1e-8,
    ),
    saveat=SaveAt([0.0, system_biref.length]),
    reltol=1e-6,
    abstol=1e-6,
)

problem = PulsePropagationProblem(; grid, system = system_biref, dofs, terms, solver, fields)

sol = solve(problem)

In [ ]:
spec_L = spectrum(sol; shifted=true, sum_modes=false);

figure(figsize=(6,2))
plot(λ, spec_L)
xlim(1500,1600)
xlabel("Wavelength (nm)")
ylabel("Intensity (a.u.)")
display(gcf())

In the example below, we explore the sensitivity structure of the output. Let's look at the intensity of the peak of the fundamental mode Raman soliton for this 10 m example. In a separate notebook.

In [ ]:
# Note that the Fourier fields that go into the solver are "unshifted" which means it's not human-plotting-friendly and so 
# we need to find the corresponding index of the shifted field.

maxind = argmax(fftshift(spec_L[:,1])) 
filter = zero(spec_L);
filter[maxind,1] = 1.0;
max_filt = filter;

weights = photon_weights(grid, f0_THz; shifted=false);

obs = FilterEnergy(; filter=max_filt .* weights, domain=:frequency, shifted=false, modes=:all)
photon_obs_peak = SpectralPhotonNumber(; filter=max_filt, modes=:all, shifted=false);
n_out = value(obs, sol)

In [ ]:
grad = gradient(
    problem,
    obs;
    trajectory=sol,
    method=PulsePropagation.Adjoint(),
    wrt=InitialField(),
    normalization=PhotonNormalized(),
    dz_adj=0.1e-2,
    reltol=1e-10,
    abstol=1e-10,
)

In [ ]:
figure(figsize=(3,3))
semilogy(λ,abs2.(fftshift(grad.initial_field_gradient,1)))
ylim(1e-5)
xlim(1500,1600)
display(gcf())

Let's now compare to automatic differentiation

In [ ]:
solver = RK4IPSolver(;
    stepping=FixedStep(dz = 1e-2),
    saveat=SaveAt([0.0, system_biref.length]),
    reltol=1e-8,
    abstol=1e-8,
)

problem = PulsePropagationProblem(; grid, system = system_biref, dofs, terms, solver, fields)

sol = solve(problem)

In [ ]:
grad_ad = gradient(
    problem,
    obs;
    trajectory=sol,
    method=AutomaticDifferentiation(),
    wrt=InitialField(),
    normalization=PhotonNormalized()
)

In [ ]:
figure(figsize=(7,7))
semilogy(λ,abs2.(fftshift(grad.initial_field_gradient,1)))
semilogy(λ,abs2.(fftshift(grad_ad.initial_field_gradient,1)),"--")
ylim(1e-5)
xlim(1500,1600)
display(gcf())

As one can see from the sensitivities, the field in mode 1 (x-polarized) is sensitive to itself and the other x-polarized spatial modes but effectively has zero sensitivity to the other polarizations. My suspicion is that the sensitivity to the y-polarized fields is "in the noise" and hence it is hard for AD and adjoint to agree on it. On the other hand, AD and adjoint agree reasonably well for the same-polarization sensitivities.